In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.ensemble import ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler

In [ ]:

# 1. 设置图形风格
sns.set(style="whitegrid")

In [ ]:
# 2. 读取数据
train = pd.read_csv("../input/train.csv")
test = pd.read_csv("../input/test.csv")


In [ ]:
print("训练集的前五行数据：")
print(train.head())

In [ ]:
print("\n训练集的缺失值情况：")
print(train.isnull().sum())

In [ ]:
print("\n测试集的缺失值情况：")
print(test.isnull().sum())


In [ ]:
# 4. 删除标准差为0的列
for col in train.columns:
    if train[col].std() == 0:
        train.drop(col, axis=1, inplace=True)
        print(f"删除标准差为0的列: {col}")


In [ ]:
# 5. 删除测试集中的冗余列
redundant_columns = ["Soil_Type7", "Soil_Type15"]
test.drop(redundant_columns, axis=1, inplace=True)
print(f"从测试集中删除冗余列: {redundant_columns}")


In [ ]:
# 6. 可视化数据分布
plt.figure(figsize=(8, 6))
sns.countplot(x='Cover_Type', data=train)
plt.title("Cover Type Distribution")
plt.xlabel("Cover Type")
plt.ylabel("Count")
plt.show()


In [ ]:
# 7. 可视化不同地理特征的散点图
plt.figure(figsize=(10, 6))
sns.scatterplot(x="Horizontal_Distance_To_Hydrology", 
                y="Vertical_Distance_To_Hydrology", 
                hue="Cover_Type", 
                data=train)
plt.title("Hydrology Distances by Cover Type")
plt.xlabel("Horizontal Distance To Hydrology")
plt.ylabel("Vertical Distance To Hydrology")
plt.legend(title="Cover Type")
plt.show()

In [ ]:
# 8. 特征工程：计算新特征
train["net_hyd_distance"] = np.sqrt(train["Vertical_Distance_To_Hydrology"]**2 + train["Horizontal_Distance_To_Hydrology"]**2)
test["net_hyd_distance"] = np.sqrt(test["Vertical_Distance_To_Hydrology"]**2 + test["Horizontal_Distance_To_Hydrology"]**2)


In [ ]:
train["mean_distance_aminity"] = (train["Horizontal_Distance_To_Hydrology"] + 
                                   train["Horizontal_Distance_To_Roadways"] + 
                                   train["Horizontal_Distance_To_Fire_Points"]) / 3


In [ ]:
test["mean_distance_aminity"] = (test["Horizontal_Distance_To_Hydrology"] + 
                                  test["Horizontal_Distance_To_Roadways"] + 
                                  test["Horizontal_Distance_To_Fire_Points"]) / 3


In [ ]:
# 10. 特征选择与分割
predictor = train.drop(["Id", "Cover_Type"], axis=1)
target = train["Cover_Type"]

In [ ]:
# 11. 特征缩放
scaler = StandardScaler()
predictor_scaled = scaler.fit_transform(predictor)

In [ ]:
# 12. 随机森林分类器训练
rf_classifier = ExtraTreesClassifier(random_state=123, n_estimators=1000)
rf_classifier.fit(predictor_scaled, target)



In [ ]:
# 13. 交叉验证评分
cv_score = cross_val_score(rf_classifier, predictor_scaled, target, cv=3).mean()
print(f"交叉验证得分: {cv_score:.4f}")



In [ ]:
# 14. 预测测试集
test_scaled = scaler.transform(test.drop("Id", axis=1))
test["Cover_Type"] = rf_classifier.predict(test_scaled)

In [ ]:
# 15. 创建提交文件
submission = test[["Id", "Cover_Type"]]
submission.to_csv("submission.csv", index=False)
print("提交文件已创建: submission.csv")

In [ ]:
# 16. 可视化结果
plt.figure(figsize=(8, 6))
sns.countplot(x="Cover_Type", data=test)
plt.title("Predicted Distribution of Cover Types")
plt.xlabel("Cover Type")
plt.ylabel("Count")
plt.show()


In [ ]:
# 17. 分类报告
predicted = cross_val_predict(rf_classifier, predictor_scaled, target, cv=3)
print(classification_report(target, predicted))


In [ ]:
# 18. 混淆矩阵可视化
conf_matrix = confusion_matrix(target, predicted)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(target), yticklabels=np.unique(target))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()